# M3 · v0.1 · Tu primer programa con un LLM

Un programa que le pregunta a Gemini sobre **tu tema**, mide tokens y tiempo, y no se cae si algo falla.

Con eso elegís una configuración y estimás el costo. Es tu **v0.1**.

**La tarea, en cuatro pasos:** 1. **Corré** tus 5 preguntas · 2. **Predecí** cuál de dos configuraciones responde mejor y gasta más (en `PREDICCION`) · 3. **Cambiá una cosa** y corré de nuevo · 4. **Compará** en la tabla.

- Todo corre acá, en Colab: no hay archivos ni carpetas.
- Corré las celdas en orden con **Shift + Enter**.
- La clave es la de la clase 1, en **Secretos** (la llave de la izquierda) como `GOOGLE_API_KEY`.

<details><summary><b>¿Sale un error?</b></summary>

- **«No pude leer el secreto GOOGLE_API_KEY»** o **`API key not valid`:** revisá el nombre del secreto (exacto, en mayúsculas),
  que «Acceso del notebook» esté activado y que la clave esté bien copiada.
- **`503 UNAVAILABLE`** («high demand»): el modelo está saturado, no es tu código. Esperá un minuto y volvé a correr.
- **`429`** (*quota*, *RESOURCE_EXHAUSTED*): se terminó la cuota gratis del día o del minuto. Esperá o seguí mañana.
- **`NameError`:** salteaste una celda o Colab se reinició. Menú *Entorno de ejecución* → *Run before*.
- **`Error controlado: …`** al final: es el error a propósito. El programa no se cae: está bien.
</details>


## 1 · Preparar (una vez, unos 30 s)

In [ ]:
%pip install -q langchain==1.4.6 langchain-google-genai==4.4.2
import os

try:
    from google.colab import userdata                      # en Colab: la clave sale de los Secretos
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass                                                   # fuera de Colab: GOOGLE_API_KEY ya tiene que estar
except Exception as error:                                 # no hay secreto con ese nombre o le falta el acceso
    print("No pude leer el secreto GOOGLE_API_KEY:", type(error).__name__)

import time

import pandas as pd
from langchain.chat_models import init_chat_model

print("Listo." if os.environ.get("GOOGLE_API_KEY") else "Falta la clave: Secretos → GOOGLE_API_KEY.")

## 2 · Perillas: lo que vas a cambiar

- `PREGUNTAS`: cambialas por **5 de tu tema** (cada una entre comillas y con coma).
- `SISTEMA`: el rol y las reglas. `MODELO` y `MAX_SALIDA`: el modelo y su tope de tokens de salida.

In [ ]:
MODELO = "google_genai:gemini-3.5-flash-lite"
SISTEMA = "Sos un asistente para estudiantes. Respondé en español, en dos frases como máximo."
PREGUNTAS = [
    "¿Qué es un token en un LLM?",
    "¿Para qué sirve la temperatura en un modelo de lenguaje?",
    "Nombrá dos riesgos de usar un LLM para tomar decisiones.",
    "¿Qué diferencia hay entre un modelo abierto y uno cerrado?",
    "¿Qué es una alucinación en un LLM?",
]
REINTENTOS = 2     # intentos en total si el proveedor falla
MAX_SALIDA = 300   # tope de tokens de salida

## 3 · El programa

- `preguntar` manda la pregunta y muestra la respuesta **de a pedazos** (*streaming*); devuelve texto, tokens y segundos.
- `registro` guarda una fila por pregunta: después la mirás como tabla.
- Si una pregunta falla (cuota, red), se reintenta, se anota y sigue.

In [ ]:
registro = []  # una fila por pregunta; cada corrida suma filas


def preguntar(modelo, pregunta):
    inicio, respuesta = time.perf_counter(), None
    for pedazo in modelo.stream([("system", SISTEMA), ("human", pregunta)]):
        print(pedazo.text, end="", flush=True)
        respuesta = pedazo if respuesta is None else respuesta + pedazo
    print()
    return respuesta.text, respuesta.usage_metadata, round(time.perf_counter() - inicio, 2)


def correr():
    modelo = init_chat_model(MODELO, max_output_tokens=MAX_SALIDA)
    for pregunta in PREGUNTAS:
        print(f"\n> {pregunta}")
        for intento in range(1, REINTENTOS + 1):
            try:
                texto, uso, segundos = preguntar(modelo, pregunta)
                print(f"  tokens={uso['input_tokens']}+{uso['output_tokens']}  latencia={segundos}s")
                registro.append({"modelo": MODELO, "sistema": SISTEMA, "pregunta": pregunta, "respuesta": texto,
                                 "entrada": uso["input_tokens"], "salida": uso["output_tokens"], "latencia_s": segundos})
                break
            except Exception as error:  # 429 (cuota), 503 (saturado), red: se reintenta y, si sigue, se anota
                print(f"  Error: {type(error).__name__} (intento {intento} de {REINTENTOS})")
                if intento < REINTENTOS:
                    time.sleep(2 * intento)
                else:
                    registro.append({"modelo": MODELO, "pregunta": pregunta, "error": type(error).__name__})

In [ ]:
correr()

**Qué deberías ver:** cada pregunta con `>`, la respuesta de a poco y `tokens=… latencia=…s`.

**Primer control · 10–15 min:** el registro tiene tokens y latencia de las 5 preguntas.

## 4 · Un error a propósito

Un modelo que no existe: el programa lo anota y no se cae. Un 404 **no** se arregla reintentando.

In [ ]:
try:
    init_chat_model("google_genai:modelo-que-no-existe").invoke("hola")
except Exception as error:
    registro.append({"modelo": "google_genai:modelo-que-no-existe", "error": type(error).__name__})
    print("Error controlado:", type(error).__name__)

## 5 · Comparar dos configuraciones

1. **Predecí:** ¿cuál responde mejor? ¿Cuál gasta más?
2. En la celda de perillas cambiá **una** cosa: `SISTEMA` con reglas («decí "no sé" si no estás seguro») o otro `MODELO`.
3. Corré esa celda y `correr()` de nuevo: el registro suma la segunda configuración.
4. Corré la tabla de abajo. Si cambiaste el `SISTEMA` (y no el modelo), poné `POR = "sistema"`.

In [ ]:
PREDICCION = ""  # antes de comparar: ¿cuál responde mejor? ¿cuál gasta más? Por ejemplo: "el SISTEMA con reglas gasta más"


In [ ]:
print("Tu predicción:", PREDICCION or "(todavía no la escribiste: hacelo antes de correr)")
POR = "modelo"  # si comparás dos SISTEMA con el mismo modelo, poné "sistema"

tabla = pd.DataFrame(registro)
bien = tabla[tabla.get("error").isna()] if "error" in tabla else tabla
print(bien.groupby(POR)[["entrada", "salida", "latencia_s"]].mean().round(1))
print(bien.groupby(POR)[["latencia_s"]].median())  # la mediana de la latencia va en tu tabla

**Qué deberías ver:** una fila por configuración con los promedios de tokens y latencia, y abajo la mediana de latencia.

## 6 · Costo por mes

Precio oficial **con fecha** ([Gemini](https://ai.google.dev/gemini-api/docs/pricing), *Paid tier*, *Standard*), en dólares por millón de tokens.

In [ ]:
PRECIO_ENTRADA = 0.30   # US$ por millón (Gemini 3.5 Flash-Lite, 19/10)
PRECIO_SALIDA = 2.50
CONSULTAS_POR_DIA = 1000

por_pregunta = bien["entrada"].mean() * PRECIO_ENTRADA / 1_000_000 + bien["salida"].mean() * PRECIO_SALIDA / 1_000_000
print(f"por pregunta: US$ {por_pregunta:.7f} · por mes: US$ {por_pregunta * CONSULTAS_POR_DIA * 30:.2f}")

## 7 · Entregar (entrega de avance 1, hasta el mar 3/11, 23:59)

- **Guardá este notebook en tu repo:** *Archivo → Guardar una copia en GitHub* → tu repositorio → `v01.ipynb`.
- **Evidencia** (una celda de texto al final, acá mismo): la tabla de las dos configuraciones, cuál elegiste y por qué, qué error se reintenta y cuál no, y el costo por mes.

**Microdefensa:** ¿qué error vale la pena reintentar y cuál no? Confianza del 1 al 4.